The following code explores a bit of how WordNet is structured, then goes on to filter all the nouns in its lexicon by certain requirements to produce a list of words which would be valid to use in the study. Consulted [Natural Language Processing with Python](https://learning.oreilly.com/library/view/natural-language-processing/9780596803346/) as well as [this paper](https://doi.org/10.1093/oxfordhb/9780199842193.001.0001) by Fellbaum.

In [13]:
# import statements
import nltk
from nltk.corpus import wordnet as wn
import numpy as np

### Exploring WordNet
First, a bit of exploring the possible relations and functionalities within WordNet. WordNet is organized by synsets, collections of synonymous words. A variety of functions then can be traversed using these synsets which make up the relational network of WordNet.

In [14]:
# look up synsets which a word appears in
wn.synsets('cabinet')

[Synset('cabinet.n.01'),
 Synset('cabinet.n.02'),
 Synset('cabinet.n.03'),
 Synset('cabinet.n.04')]

In [19]:
# bit of code which prints the definition for each synset a word is part of
word = 'pant'
for s in wn.synsets(word, pos=wn.NOUN):
    name = s.name()
    print(name + ":", wn.synset(name).definition())

slide_fastener.n.01: a fastener for locking together two toothed edges by means of a sliding tab


In [20]:
# see lemmas (words which actually make up synset)
wn.synset('car.n.01').lemma_names()

['slide_fastener', 'zip', 'zipper', 'zip_fastener']

In [6]:
# WordNet links together concepts in a hierarchy, can navigate between concepts
# hypernym is more general, hyponym is more specific
wn.synset('car.n.01').hypernyms() #immediate hypernyms
wn.synset('car.n.01').hyponyms() # immediate hyponyms

[Synset('ambulance.n.01'),
 Synset('beach_wagon.n.01'),
 Synset('bus.n.04'),
 Synset('cab.n.03'),
 Synset('compact.n.03'),
 Synset('convertible.n.01'),
 Synset('coupe.n.01'),
 Synset('cruiser.n.01'),
 Synset('electric.n.01'),
 Synset('gas_guzzler.n.01'),
 Synset('hardtop.n.01'),
 Synset('hatchback.n.01'),
 Synset('horseless_carriage.n.01'),
 Synset('hot_rod.n.01'),
 Synset('jeep.n.01'),
 Synset('limousine.n.01'),
 Synset('loaner.n.02'),
 Synset('minicar.n.01'),
 Synset('minivan.n.01'),
 Synset('model_t.n.01'),
 Synset('pace_car.n.01'),
 Synset('racer.n.02'),
 Synset('roadster.n.01'),
 Synset('sedan.n.01'),
 Synset('sport_utility.n.01'),
 Synset('sports_car.n.01'),
 Synset('stanley_steamer.n.01'),
 Synset('stock_car.n.01'),
 Synset('subcompact.n.01'),
 Synset('touring_car.n.01'),
 Synset('used-car.n.01')]

In [7]:
# can also find the lowest common hypernym of two synsets
wn.synset('chair.n.01').lowest_common_hypernyms(wn.synset('table.n.02'))

[Synset('furniture.n.01')]

### Filtering the noun set
For our purposes, we want to posit certain restrictions in line with our study design and filter the set of all the nouns in WordNet to get only ones which meet these restrictions.

In [8]:
# these functions check for the relevant qualities we want each word to meet
def has_enough_synonyms(synset):
    return len(synset.lemma_names()) > 1

def has_enough_meronyms(synset):
    return len(synset.part_meronyms()) > 1

def has_sister(synset):
    for hyp in synset.hypernyms():
        if len(hyp.hyponyms()) > 2:
            return True
    return False

In [9]:
# returns a string in the desired format to output to file
# string is formatted as [line1, line2, ...] to be compatible with writelines()
def get_candidate_string(synset):
    candidate_string = []
    candidate_string.append("X: " + synset.name() + '\n')
    candidate_string.append("L: " + ", ".join(x for x in synset.lemma_names()) + '\n')
    candidate_string.append("M: " + ", ".join(x.name() for x in synset.part_meronyms()) + '\n')
    candidate_string.append("H: " + ", ".join(x.name() for x in synset.hypernyms()) + '\n')
    for item in ["T: " + item for item in [", ".join(sublist) for sublist in [[y.name() for y in hypernym] for hypernym in [x.hyponyms() for x in synset.hypernyms()]]]]:
        candidate_string.append(item + '\n')
    candidate_string.append('\n')
    return candidate_string


In [10]:
# this will write candidate nouns to the output file candidates.txt!
with open('data/candidates.txt', 'w') as f:
    for synset in wn.all_synsets('n'):
        if has_enough_synonyms(synset) and has_enough_meronyms(synset) and has_sister(synset):
            f.writelines(get_candidate_string(synset))
    

In [11]:
# for checking the lemmas of multiple synsets at once
name_list = ['case.n.05', 'housing.n.01']
for name in name_list:
    print(name + ': ', end='')
    print(wn.synset(name).lemma_names())

case.n.05: ['case']
housing.n.01: ['housing', 'lodging', 'living_accommodations']
